# P-tuning 实战

## Step1 导入相关包

In [1]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

## Step2 加载数据集

In [2]:
ds = Dataset.load_from_disk("../data/alpaca_data_zh/")
ds

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 26858
})

In [3]:
ds[:3]

{'output': ['以下是保持健康的三个提示：\n\n1. 保持身体活动。每天做适当的身体运动，如散步、跑步或游泳，能促进心血管健康，增强肌肉力量，并有助于减少体重。\n\n2. 均衡饮食。每天食用新鲜的蔬菜、水果、全谷物和脂肪含量低的蛋白质食物，避免高糖、高脂肪和加工食品，以保持健康的饮食习惯。\n\n3. 睡眠充足。睡眠对人体健康至关重要，成年人每天应保证 7-8 小时的睡眠。良好的睡眠有助于减轻压力，促进身体恢复，并提高注意力和记忆力。',
  '4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。',
  '朱利叶斯·凯撒，又称尤利乌斯·恺撒（Julius Caesar）是古罗马的政治家、军事家和作家。他于公元前44年3月15日被刺杀。 \n\n根据历史记载，当时罗马元老院里一些参议员联合起来策划了对恺撒的刺杀行动，因为他们担心恺撒的统治将给罗马共和制带来威胁。在公元前44年3月15日（又称“3月的艾达之日”），恺撒去参加元老院会议时，被一群参议员包围并被攻击致死。据记载，他身中23刀，其中一刀最终致命。'],
 'input': ['', '输入：4/16', ''],
 'instruction': ['保持健康的三个提示。', '解释为什么以下分数等同于1/4', '朱利叶斯·凯撒是如何死亡的？']}

## Step3 数据集预处理

In [4]:
tokenizer = AutoTokenizer.from_pretrained("../../model/langboat/bloom-1b4-zh")
tokenizer

BloomTokenizerFast(name_or_path='../../model/langboat/bloom-1b4-zh', vocab_size=46145, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [5]:
def process_func(example):
    MAX_LENGTH = 256
    input_ids, attention_mask, labels = [], [], []
    instruction = tokenizer("\n".join(["Human: " + example["instruction"], example["input"]]).strip() + "\n\nAssistant: ")
    response = tokenizer(example["output"] + tokenizer.eos_token)
    input_ids = instruction["input_ids"] + response["input_ids"]
    attention_mask = instruction["attention_mask"] + response["attention_mask"]
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"]
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [6]:
tokenized_ds = ds.map(process_func, remove_columns=ds.column_names)
tokenized_ds

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 26858
})

In [7]:
tokenizer.decode(tokenized_ds[1]["input_ids"])

'Human: 解释为什么以下分数等同于1/4\n输入：4/16\n\nAssistant: 4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。</s>'

In [8]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_ds[1]["labels"])))

'4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。</s>'

## Step4 创建模型

In [9]:
model = AutoModelForCausalLM.from_pretrained("../../model/langboat/bloom-1b4-zh")

In [10]:
model = model.cuda()
ipt = tokenizer("Human: {}\n{}".format("数学考试有哪些技巧？", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
print(tokenizer.decode(model.generate(**ipt, max_length=256, do_sample=True)[0], skip_special_tokens=True))

Human: 数学考试有哪些技巧？

Assistant: 数学考试有很多要考虑的
包括以下几点： 1. 考试的科目是什么
2. 本科考试是什么科？
3. 什么时候准备？
4. 学习该科有哪些要点？
5. 学习方法有哪些？
6. 学习心态有哪些？
7. 学习兴趣有哪些？
8. 学习方法有哪些？
9. 如何和成绩较差的同学会？
10. 学习有什么特点？
11. 数学基础怎么样？
12. 数学基础薄弱怎么办？
13. 数学基础不强，怎么办？
15. 学习中遇到的问题怎么办？
16. 数学学习应该注意点是什么？
17. 数学学习有什么困惑？
18. 数学学习应该注意什么？
19. 怎样与家长沟通？
20. 怎样与老师沟通？
21. 数学学习的方法有哪些？
22. 数学学习有什么窍门？
23. 数学学习有什么忌讳？
24. 上大学后，数学成绩如何提高？
25. 怎样克服数学学习中遇到的困难？
26. 提高数学成绩的经验有哪些？
27. 如何提高


## P-tuning

### PEFT Step1 配置文件

In [11]:
from peft import PromptEncoderConfig, TaskType, get_peft_model, PromptEncoderReparameterizationType

config = PromptEncoderConfig(task_type=TaskType.CAUSAL_LM, num_virtual_tokens=10,
                             encoder_reparameterization_type=PromptEncoderReparameterizationType.MLP,
                             encoder_dropout=0.1, encoder_num_layers=5, encoder_hidden_size=1024)
config

PromptEncoderConfig(peft_type=<PeftType.P_TUNING: 'P_TUNING'>, auto_mapping=None, base_model_name_or_path=None, revision=None, task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, inference_mode=False, num_virtual_tokens=10, token_dim=None, num_transformer_submodules=None, num_attention_heads=None, num_layers=None, encoder_reparameterization_type=<PromptEncoderReparameterizationType.MLP: 'MLP'>, encoder_hidden_size=1024, encoder_num_layers=5, encoder_dropout=0.1)

### PEFT Step2 创建模型

In [12]:
model = get_peft_model(model, config)

/home/mizuki/anaconda3/envs/trans/lib/python3.9/site-packages/peft/tuners/p_tuning/model.py:105: UserWarning: for MLP, the argument `encoder_num_layers` is ignored. Exactly 2 MLP layers are used.
  warnings.warn(


In [13]:
config

PromptEncoderConfig(peft_type=<PeftType.P_TUNING: 'P_TUNING'>, auto_mapping=None, base_model_name_or_path='../../model/langboat/bloom-1b4-zh', revision=None, task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, inference_mode=False, num_virtual_tokens=10, token_dim=2048, num_transformer_submodules=1, num_attention_heads=16, num_layers=24, encoder_reparameterization_type=<PromptEncoderReparameterizationType.MLP: 'MLP'>, encoder_hidden_size=1024, encoder_num_layers=5, encoder_dropout=0.1)

In [14]:
model.print_trainable_parameters()

trainable params: 5,267,456 || all params: 1,308,379,136 || trainable%: 0.4026


## Step5 配置训练参数

In [15]:
args = TrainingArguments(
    output_dir="./chatbot",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    logging_steps=10,
    num_train_epochs=1
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## Step6 创建训练器

In [16]:
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

## Step7 模型训练

In [17]:
trainer.train()

  0%|          | 0/3357 [00:00<?, ?it/s]

{'loss': 2.9186, 'grad_norm': 2.902909278869629, 'learning_rate': 4.985105749180816e-05, 'epoch': 0.0}
{'loss': 2.8641, 'grad_norm': 6.197839736938477, 'learning_rate': 4.9702114983616324e-05, 'epoch': 0.01}
{'loss': 2.659, 'grad_norm': 3.0021495819091797, 'learning_rate': 4.9553172475424484e-05, 'epoch': 0.01}
{'loss': 2.6358, 'grad_norm': 4.437957763671875, 'learning_rate': 4.940422996723265e-05, 'epoch': 0.01}
{'loss': 2.6378, 'grad_norm': 3.1732568740844727, 'learning_rate': 4.925528745904081e-05, 'epoch': 0.01}
{'loss': 2.6417, 'grad_norm': 1.626693606376648, 'learning_rate': 4.910634495084897e-05, 'epoch': 0.02}
{'loss': 2.4738, 'grad_norm': 2.357253074645996, 'learning_rate': 4.895740244265713e-05, 'epoch': 0.02}
{'loss': 2.5333, 'grad_norm': 2.356311798095703, 'learning_rate': 4.88084599344653e-05, 'epoch': 0.02}
{'loss': 2.4173, 'grad_norm': 3.7707202434539795, 'learning_rate': 4.865951742627346e-05, 'epoch': 0.03}
{'loss': 2.3976, 'grad_norm': 1.5840569734573364, 'learning_ra

/home/mizuki/anaconda3/envs/trans/lib/python3.9/site-packages/peft/utils/save_and_load.py:195: UserWarning: Could not find a config file in ../../model/langboat/bloom-1b4-zh - will assume that the vocabulary was not modified.
  warnings.warn(


{'loss': 2.3371, 'grad_norm': 2.091146230697632, 'learning_rate': 4.240393208221627e-05, 'epoch': 0.15}
{'loss': 2.5371, 'grad_norm': 1.2338289022445679, 'learning_rate': 4.225498957402443e-05, 'epoch': 0.15}
{'loss': 2.3497, 'grad_norm': 2.299574613571167, 'learning_rate': 4.2106047065832595e-05, 'epoch': 0.16}
{'loss': 2.382, 'grad_norm': 2.115823268890381, 'learning_rate': 4.1957104557640756e-05, 'epoch': 0.16}
{'loss': 2.3677, 'grad_norm': 1.5625663995742798, 'learning_rate': 4.180816204944891e-05, 'epoch': 0.16}
{'loss': 2.3265, 'grad_norm': 2.0544381141662598, 'learning_rate': 4.165921954125708e-05, 'epoch': 0.17}
{'loss': 2.4168, 'grad_norm': 1.436777949333191, 'learning_rate': 4.151027703306524e-05, 'epoch': 0.17}
{'loss': 2.4706, 'grad_norm': 4.205395698547363, 'learning_rate': 4.1361334524873405e-05, 'epoch': 0.17}
{'loss': 2.3397, 'grad_norm': 1.5016584396362305, 'learning_rate': 4.121239201668156e-05, 'epoch': 0.18}
{'loss': 2.3055, 'grad_norm': 2.099501132965088, 'learning

/home/mizuki/anaconda3/envs/trans/lib/python3.9/site-packages/peft/utils/save_and_load.py:195: UserWarning: Could not find a config file in ../../model/langboat/bloom-1b4-zh - will assume that the vocabulary was not modified.
  warnings.warn(


...

{'loss': 2.2236, 'grad_norm': 4.877227306365967, 'learning_rate': 1.2615430443848674e-05, 'epoch': 0.75}
{'loss': 2.2305, 'grad_norm': 3.553398847579956, 'learning_rate': 1.2466487935656837e-05, 'epoch': 0.75}
{'loss': 2.354, 'grad_norm': 2.3621058464050293, 'learning_rate': 1.2317545427464999e-05, 'epoch': 0.75}
{'loss': 2.3392, 'grad_norm': 1.5690864324569702, 'learning_rate': 1.2168602919273161e-05, 'epoch': 0.76}
{'loss': 2.2465, 'grad_norm': 2.3077661991119385, 'learning_rate': 1.2019660411081324e-05, 'epoch': 0.76}
{'loss': 2.2505, 'grad_norm': 1.2192528247833252, 'learning_rate': 1.1870717902889484e-05, 'epoch': 0.76}
{'loss': 2.3045, 'grad_norm': 9.478109359741211, 'learning_rate': 1.1721775394697646e-05, 'epoch': 0.77}
{'loss': 2.3842, 'grad_norm': 1.8584450483322144, 'learning_rate': 1.1572832886505809e-05, 'epoch': 0.77}
{'loss': 2.4414, 'grad_norm': 3.61769700050354, 'learning_rate': 1.1423890378313971e-05, 'epoch': 0.77}
{'loss': 2.3386, 'grad_norm': 2.637331485748291, 'le

/home/mizuki/anaconda3/envs/trans/lib/python3.9/site-packages/peft/utils/save_and_load.py:195: UserWarning: Could not find a config file in ../../model/langboat/bloom-1b4-zh - will assume that the vocabulary was not modified.
  warnings.warn(


{'loss': 2.2399, 'grad_norm': 1.8426374197006226, 'learning_rate': 5.168305034256777e-06, 'epoch': 0.9}
{'loss': 2.2915, 'grad_norm': 2.6994588375091553, 'learning_rate': 5.019362526064939e-06, 'epoch': 0.9}
{'loss': 2.3198, 'grad_norm': 2.566716194152832, 'learning_rate': 4.870420017873101e-06, 'epoch': 0.9}
{'loss': 2.2106, 'grad_norm': 1.7766456604003906, 'learning_rate': 4.721477509681263e-06, 'epoch': 0.91}
{'loss': 2.3084, 'grad_norm': 2.026052474975586, 'learning_rate': 4.572535001489425e-06, 'epoch': 0.91}
{'loss': 2.2545, 'grad_norm': 1.6475948095321655, 'learning_rate': 4.423592493297587e-06, 'epoch': 0.91}
{'loss': 2.4087, 'grad_norm': 1.3471111059188843, 'learning_rate': 4.274649985105749e-06, 'epoch': 0.91}
{'loss': 2.3638, 'grad_norm': 1.6876273155212402, 'learning_rate': 4.125707476913911e-06, 'epoch': 0.92}
{'loss': 2.3589, 'grad_norm': 1.8587669134140015, 'learning_rate': 3.976764968722073e-06, 'epoch': 0.92}
{'loss': 2.2175, 'grad_norm': 1.5614343881607056, 'learning_

/home/mizuki/anaconda3/envs/trans/lib/python3.9/site-packages/peft/utils/save_and_load.py:195: UserWarning: Could not find a config file in ../../model/langboat/bloom-1b4-zh - will assume that the vocabulary was not modified.
  warnings.warn(


TrainOutput(global_step=3357, training_loss=2.33290943149161, metrics={'train_runtime': 3203.598, 'train_samples_per_second': 8.384, 'train_steps_per_second': 1.048, 'total_flos': 1.476411489067008e+16, 'train_loss': 2.33290943149161, 'epoch': 0.9999255342914588})

## Step8 模型推理

In [18]:
model = model.cuda()
ipt = tokenizer("Human: {}\n{}".format("数学考试有哪些技巧？", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
print(tokenizer.decode(model.generate(**ipt, max_length=256, do_sample=True)[0], skip_special_tokens=True))

Human: 数学考试有哪些技巧？

Assistant: 数学考试是一个较为理性的任务，而除了具备逻辑清晰和理解复杂数学概念的能力外，一些心理技巧也非常关键。下面是一些可以帮助你更好应付数学考试的心理技巧：

1. 树立明确的目标
在制定目标的时候，首先需要了解自己的能力水平和知识层次，不要盲目的制定过高或者过低的考试目标。制定明确的考试目标和时间框架，可以帮助你更好规划复习时间和计划。

2. 调整情绪
在数学考试中，大部分题目的难度都比较大，要学会用适当的策略来应对数学难题，调整好自己的心情和状态，才能在考试时发挥好的水平。

3. 保持积极乐观的态度
当数学考试遇到困难时，你也许会感到焦虑，或者感到沮丧。要学会用适时恰当的自我暗示来减轻心理压力，保持乐观积极的人生态度，相信自己，相信自己的数学能力。

4. 寻找挑战
通过做挑战性题目，你可以获得对数学概念、难题及知识点的理解、记忆和思考能力。此外，在做练习题的过程中，你还可以加深对数学知识的印象，提高对数学内容的学习兴趣。


